In [1]:
import math
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use NLTK for tokenization and stopword removal.
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    # Fallback keeps the notebook usable if the NLTK stopwords corpus
    # has not yet been downloaded in the execution environment.
    STOP_WORDS = {
        "a","an","and","are","as","at","be","been","by","for","from",
        "has","have","he","her","his","i","if","in","is","it","its",
        "me","my","of","on","or","our","she","that","the","their",
        "them","there","these","they","this","to","was","we","were",
        "what","when","where","which","who","will","with","you","your"
    }

def preprocess(text):
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    return [t for t in tokens if t not in STOP_WORDS]

def clean_text(text):
    return " ".join(preprocess(text))

print("Libraries loaded successfully.")
print("Stopwords available:", len(STOP_WORDS))


Libraries loaded successfully.
Stopwords available: 198


In [2]:
# Q1(a): Create a corpus containing 5 documents.
documents = [
    "Machine learning helps computers learn patterns from data. Learning algorithms improve predictions.",
    "Natural language processing helps computers understand human language. Text data is processed using language models.",
    "Deep learning uses neural networks to learn complex patterns. Neural models are trained on large datasets.",
    "Data science combines statistics, programming, and machine learning. Data analysis supports better decisions.",
    "Computer vision enables computers to understand images and videos. Deep learning models recognize visual patterns."
]

processed = [preprocess(doc) for doc in documents]

for i, (original, tokens) in enumerate(zip(documents, processed), 1):
    print(f"Document {i}")
    print("Original :", original)
    print("Processed:", tokens)
    print()


Document 1
Original : Machine learning helps computers learn patterns from data. Learning algorithms improve predictions.
Processed: ['machine', 'learning', 'helps', 'computers', 'learn', 'patterns', 'data', 'learning', 'algorithms', 'improve', 'predictions']

Document 2
Original : Natural language processing helps computers understand human language. Text data is processed using language models.
Processed: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language', 'text', 'data', 'processed', 'using', 'language', 'models']

Document 3
Original : Deep learning uses neural networks to learn complex patterns. Neural models are trained on large datasets.
Processed: ['deep', 'learning', 'uses', 'neural', 'networks', 'learn', 'complex', 'patterns', 'neural', 'models', 'trained', 'large', 'datasets']

Document 4
Original : Data science combines statistics, programming, and machine learning. Data analysis supports better decisions.
Processed: ['data', 'scie

In [3]:
# Q1(c): Construct the corpus vocabulary.
vocabulary = sorted(set(token for doc in processed for token in doc))

print("Vocabulary size:", len(vocabulary))
print(vocabulary)


Vocabulary size: 42
['algorithms', 'analysis', 'better', 'combines', 'complex', 'computer', 'computers', 'data', 'datasets', 'decisions', 'deep', 'enables', 'helps', 'human', 'images', 'improve', 'language', 'large', 'learn', 'learning', 'machine', 'models', 'natural', 'networks', 'neural', 'patterns', 'predictions', 'processed', 'processing', 'programming', 'recognize', 'science', 'statistics', 'supports', 'text', 'trained', 'understand', 'uses', 'using', 'videos', 'vision', 'visual']


In [4]:
# Q1(d): Calculate Term Frequency for every term in every document.
tf_matrix = pd.DataFrame(
    0.0,
    index=[f"Document {i}" for i in range(1, len(documents) + 1)],
    columns=vocabulary
)

for i, tokens in enumerate(processed):
    counts = pd.Series(tokens).value_counts()
    total = len(tokens)
    for term in vocabulary:
        tf_matrix.iloc[i, tf_matrix.columns.get_loc(term)] = counts.get(term, 0) / total

print("TF Matrix")
display(tf_matrix.round(4))
print("Dimensions:", tf_matrix.shape)


TF Matrix


,algorithms,analysis,better,combines,complex,computer,computers,data,datasets,decisions,...,statistics,supports,text,trained,understand,uses,using,videos,vision,visual
Document 1,0.0909,0.0000,0.0000,0.0000,0.0000,0.0000,0.0909,0.0909,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Document 2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0714,0.0714,0.0000,0.0000,...,0.0000,0.0000,0.0714,0.0000,0.0714,0.0000,0.0714,0.0000,0.0000,0.0000
Document 3,0.0000,0.0000,0.0000,0.0000,0.0769,0.0000,0.0000,0.0000,0.0769,0.0000,...,0.0000,0.0000,0.0000,0.0769,0.0000,0.0769,0.0000,0.0000,0.0000,0.0000
Document 4,0.0000,0.0833,0.0833,0.0833,0.0000,0.0000,0.0000,0.1667,0.0000,0.0833,...,0.0833,0.0833,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Document 5,0.0000,0.0000,0.0000,0.0000,0.0000,0.0769,0.0769,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0769,0.0000,0.0000,0.0769,0.0769,0.0769


Dimensions: (5, 42)


In [5]:
# Q1(e): Calculate IDF for every vocabulary term.
N = len(processed)

document_frequency = {
    term: sum(term in tokens for tokens in processed)
    for term in vocabulary
}

idf = pd.Series(
    {term: math.log(N / document_frequency[term]) for term in vocabulary},
    name="IDF"
)

print("IDF Values")
display(idf.to_frame().round(4))


IDF Values


,IDF
algorithms,1.6094
analysis,1.6094
better,1.6094
combines,1.6094
complex,1.6094
computer,1.6094
computers,0.5108
data,0.5108
datasets,1.6094
decisions,1.6094


In [6]:
# Q1(f): Calculate TF-IDF = TF × IDF.
tfidf_matrix = tf_matrix.mul(idf, axis=1)

print("TF-IDF Matrix")
display(tfidf_matrix.round(4))
print("Dimensions:", tfidf_matrix.shape)


TF-IDF Matrix


,algorithms,analysis,better,combines,complex,computer,computers,data,datasets,decisions,...,statistics,supports,text,trained,understand,uses,using,videos,vision,visual
Document 1,0.1463,0.0000,0.0000,0.0000,0.0000,0.0000,0.0464,0.0464,0.0000,0.0000,...,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000
Document 2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0365,0.0365,0.0000,0.0000,...,0.0000,0.0000,0.115,0.0000,0.0654,0.0000,0.115,0.0000,0.0000,0.0000
Document 3,0.0000,0.0000,0.0000,0.0000,0.1238,0.0000,0.0000,0.0000,0.1238,0.0000,...,0.0000,0.0000,0.000,0.1238,0.0000,0.1238,0.000,0.0000,0.0000,0.0000
Document 4,0.0000,0.1341,0.1341,0.1341,0.0000,0.0000,0.0000,0.0851,0.0000,0.1341,...,0.1341,0.1341,0.000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000
Document 5,0.0000,0.0000,0.0000,0.0000,0.0000,0.1238,0.0393,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.000,0.0000,0.0705,0.0000,0.000,0.1238,0.1238,0.1238


Dimensions: (5, 42)


In [7]:
# Q1(g): Display the final required representations.
print("Vocabulary:", vocabulary)
print("\nTF Matrix:")
display(tf_matrix.round(4))

print("IDF Values:")
display(idf.to_frame().round(4))

print("TF-IDF Matrix:")
display(tfidf_matrix.round(4))


Vocabulary: ['algorithms', 'analysis', 'better', 'combines', 'complex', 'computer', 'computers', 'data', 'datasets', 'decisions', 'deep', 'enables', 'helps', 'human', 'images', 'improve', 'language', 'large', 'learn', 'learning', 'machine', 'models', 'natural', 'networks', 'neural', 'patterns', 'predictions', 'processed', 'processing', 'programming', 'recognize', 'science', 'statistics', 'supports', 'text', 'trained', 'understand', 'uses', 'using', 'videos', 'vision', 'visual']

TF Matrix:


,algorithms,analysis,better,combines,complex,computer,computers,data,datasets,decisions,...,statistics,supports,text,trained,understand,uses,using,videos,vision,visual
Document 1,0.0909,0.0000,0.0000,0.0000,0.0000,0.0000,0.0909,0.0909,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Document 2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0714,0.0714,0.0000,0.0000,...,0.0000,0.0000,0.0714,0.0000,0.0714,0.0000,0.0714,0.0000,0.0000,0.0000
Document 3,0.0000,0.0000,0.0000,0.0000,0.0769,0.0000,0.0000,0.0000,0.0769,0.0000,...,0.0000,0.0000,0.0000,0.0769,0.0000,0.0769,0.0000,0.0000,0.0000,0.0000
Document 4,0.0000,0.0833,0.0833,0.0833,0.0000,0.0000,0.0000,0.1667,0.0000,0.0833,...,0.0833,0.0833,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Document 5,0.0000,0.0000,0.0000,0.0000,0.0000,0.0769,0.0769,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0769,0.0000,0.0000,0.0769,0.0769,0.0769


IDF Values:


,IDF
algorithms,1.6094
analysis,1.6094
better,1.6094
combines,1.6094
complex,1.6094
computer,1.6094
computers,0.5108
data,0.5108
datasets,1.6094
decisions,1.6094


TF-IDF Matrix:


,algorithms,analysis,better,combines,complex,computer,computers,data,datasets,decisions,...,statistics,supports,text,trained,understand,uses,using,videos,vision,visual
Document 1,0.1463,0.0000,0.0000,0.0000,0.0000,0.0000,0.0464,0.0464,0.0000,0.0000,...,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000
Document 2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0365,0.0365,0.0000,0.0000,...,0.0000,0.0000,0.115,0.0000,0.0654,0.0000,0.115,0.0000,0.0000,0.0000
Document 3,0.0000,0.0000,0.0000,0.0000,0.1238,0.0000,0.0000,0.0000,0.1238,0.0000,...,0.0000,0.0000,0.000,0.1238,0.0000,0.1238,0.000,0.0000,0.0000,0.0000
Document 4,0.0000,0.1341,0.1341,0.1341,0.0000,0.0000,0.0000,0.0851,0.0000,0.1341,...,0.1341,0.1341,0.000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000
Document 5,0.0000,0.0000,0.0000,0.0000,0.0000,0.1238,0.0393,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.000,0.0000,0.0705,0.0000,0.000,0.1238,0.1238,0.1238


In [8]:
# Q1(h): Identify the most important terms in each document.
# Terms with larger TF-IDF values are more characteristic of that document.

for doc in tfidf_matrix.index:
    top_terms = tfidf_matrix.loc[doc].sort_values(ascending=False).head(5)
    print(f"\n{doc}:")
    for term, score in top_terms.items():
        print(f"  {term:<15} {score:.4f}")



Document 1:
  algorithms      0.1463
  improve         0.1463
  predictions     0.1463
  machine         0.0833
  helps           0.0833

Document 2:
  language        0.3449
  human           0.1150
  processed       0.1150
  processing      0.1150
  using           0.1150

Document 3:
  neural          0.2476
  datasets        0.1238
  uses            0.1238
  networks        0.1238
  large           0.1238

Document 4:
  analysis        0.1341
  better          0.1341
  decisions       0.1341
  combines        0.1341
  statistics      0.1341

Document 5:
  computer        0.1238
  enables         0.1238
  recognize       0.1238
  videos          0.1238
  visual          0.1238


### Observation

TF measures how frequently a term occurs within a document. IDF reduces the importance of terms that occur across many documents. TF-IDF therefore gives larger weights to terms that are relatively characteristic of a particular document.
